<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/01_ingesta/02_ENARES_2024_STAGE1_data_ingestion_inei.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENARES 2024 CRS04 ML Pipeline

## Stage 1 — Data Ingestion

### Notebook Information

| Field | Value |
|---|---|
| **Notebook name** | `02_ENARES_2024_STAGE1_data_ingestion_inei.ipynb` |
| **Role** | Computer Science Lead — Data Engineering |
| **Author** | Ana Cordero Ricaldi |
| **Account used** | `anacordero.001@gmail.com` |
| **Official source** | INEI Microdatos portal |
| **Official source URL** | <https://proyectos.inei.gob.pe/microdatos/> |
| **Selected format** | SPSS ZIP |
| **Modules covered** | `976-Modulo1941` to `976-Modulo1962` |
| **Script version** | `stage1-ingestion-v2.0` |
| **Date executed** | `[AUTO-GENERATED AT RUNTIME]` |

---

# OBJECTIVE

Download or reuse the official ENARES 2024 SPSS ZIP packages from the INEI Microdatos portal, preserve the original ZIP files intact, extract the included `.sav` and PDF files, calculate SHA-256 checksums, and generate the Stage 1 manifest, ingestion log, and raw module catalogue.

---

# METHODOLOGICAL DECISION

SPSS ZIP is selected as the official raw source format because `.sav` files preserve:

- variable labels;
- value labels;
- coding metadata required for reproducible interpretation.

CSV and Stata formats are acknowledged as available alternatives but are not selected as the primary source format.

---

# STRICT LIMITS OF THIS NOTEBOOK

This notebook only performs **Stage 1 ingestion**.

It strictly **DOES NOT** perform:

- BigQuery loading;
- cleaning;
- recoding;
- merging;
- statistical analysis;
- visualisation;
- modelling.

---

# EXPECTED OUTPUTS

```text
01BasesDatosPrimarias/
└── ENARES_2024_STAGE1_manifest_YYYYMMDD_HHMMSS.json

05Resultados/logs/
├── ENARES_2024_STAGE1_log_ingesta_YYYYMMDD_HHMMSS.txt
└── ENARES_2024_STAGE1_catalogo_modulos.csv
```

In [1]:
!pip install pyreadstat tqdm google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 25.1 MB/s eta 0:00:00


In [2]:

import os
import json
import hashlib
import requests
import zipfile
import time
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from google.colab import auth
from googleapiclient.discovery import build
from google.colab import drive


In [3]:
drive.mount('/content/drive')
auth.authenticate_user()
drive_service = build("drive", "v3")

SCRIPT_VERSION = "stage1-ingestion-v2.0"
OFFICIAL_MICRODATA_PORTAL = "https://proyectos.inei.gob.pe/microdatos/"
OFFICIAL_SELECTED_PACKAGE_TYPE = "SPSS ZIP"
SELECTED_DOWNLOAD_FORMAT = "SPSS"
ALTERNATIVE_FORMATS_AVAILABLE = ["CSV ZIP", "Stata ZIP"]
ALTERNATIVE_FORMATS_NOT_SELECTED_REASON = (
    "SPSS ZIP was selected because .sav preserves variable labels, value labels, "
    "and coding metadata required for reproducible interpretation. CSV/Stata were "
    "not selected as primary source formats."
)

FORCE_REEXTRACT = False # Guard against unnecessary re-extraction

ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"
RAW_DIR = f"{ROOT}/01BasesDatosPrimarias"
LOG_DIR = f"{ROOT}/05Resultados/logs"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive


In [4]:
_drive_cache = {}

def _drive_escape(value):
    # Safely escapes backslashes and apostrophes for Drive API queries
    return value.replace("\\", "\\\\").replace("'", "\\'")

def find_drive_child(parent_id, name):
    cache_key = (parent_id, name)
    if cache_key in _drive_cache:
        return _drive_cache[cache_key]

    query = f"name = '{_drive_escape(name)}' and '{parent_id}' in parents and trashed = false"
    response = drive_service.files().list(
        q=query,
        fields="files(id,name,mimeType,size,webViewLink)",
        pageSize=10,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()

    files = response.get("files", [])
    item = files[0] if files else None
    _drive_cache[cache_key] = item
    return item

def drive_item_for_path(path):
    """Return Drive metadata for a path under /content/drive/MyDrive, if found."""
    rel_path = os.path.relpath(path, "/content/drive/MyDrive")
    if rel_path.startswith(".."):
        return None

    parts = [p for p in rel_path.replace("\\", "/").split("/") if p]
    parent_id = "root"
    item = None

    for part in parts:
        item = find_drive_child(parent_id, part)
        if item is None:
            return None
        parent_id = item["id"]

    return item

def drive_id_for_path_with_retry(path, retries=5, wait_seconds=2):
    """Retrieves Drive ID with retry logic to account for sync delays."""
    for attempt in range(retries):
        item = drive_item_for_path(path)
        if item:
            return item.get("id")
        time.sleep(wait_seconds)
    return None

In [5]:
def sha256_file(path):
    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

def download_file(url, output_path, retries=3):
    if os.path.exists(output_path):
        return "cached"

    for attempt in range(retries):
        try:
            r = requests.get(url, stream=True, timeout=30)
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            return "downloaded"

        except Exception as e:
            time.sleep(2 ** attempt)
    raise Exception(f"Failed to download: {url}")

def extract_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

In [6]:
base_url = "https://proyectos.inei.gob.pe/iinei/srienaho/descarga/SPSS/976-Modulo{}.zip"
modules = [{"module": f"Modulo{i}", "url": base_url.format(i)} for i in range(1941, 1963)]

manifest = []
catalog = []
log_lines = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

log_lines.append(f"=== ENARES 2024 STAGE 1 INGESTION EXECUTED AT {timestamp} ===")

In [7]:
print("Starting ingestion loop...")
for m in tqdm(modules, desc="Processing Modules"):
    module_name = m["module"]
    module_number = module_name.replace("Modulo", "")
    module_id = f"976-Modulo{module_number}"
    url = m["url"]

    log_lines.append(f"\n--- MODULE: {module_id} ---")
    log_lines.append(f"SOURCE URL: {url}")

    module_folder = os.path.join(RAW_DIR, module_name)
    os.makedirs(module_folder, exist_ok=True)
    drive_folder_id = drive_id_for_path_with_retry(module_folder)

    zip_path = os.path.join(module_folder, f"{module_name}.zip")
    log_lines.append(f"ZIP PATH: {zip_path}")

    # A. Download
    try:
        dl_status = download_file(url, zip_path)
        log_lines.append(f"DOWNLOAD STATUS: {'Reused from cache' if dl_status == 'cached' else 'Newly downloaded'}")
    except Exception as e:
        log_lines.append(f"STATUS: FAILED_DOWNLOAD")
        log_lines.append(f"ERROR: {str(e)}")
        manifest.append({
            "module_id": module_id,
            "status": "failed_download",
            "error": str(e)
        })
        continue

    zip_hash = sha256_file(zip_path)
    zip_size = os.path.getsize(zip_path)
    zip_drive_id = drive_id_for_path_with_retry(zip_path)

    log_lines.append(f"ZIP SHA256: {zip_hash}")
    log_lines.append(f"ZIP SIZE BYTES: {zip_size}")

    # B. Extract Guard Logic
    extract_folder = os.path.join(module_folder, "extracted")
    os.makedirs(extract_folder, exist_ok=True)

    is_empty = not any(os.scandir(extract_folder))
    if FORCE_REEXTRACT or is_empty:
        extract_zip(zip_path, extract_folder)
        log_lines.append(f"EXTRACTION STATUS: Newly extracted")
    else:
        log_lines.append(f"EXTRACTION STATUS: Reused from cache")

    # C. File Cataloging & PDF Classification
    extracted_records = []
    for root, _, files in os.walk(extract_folder):
        for file in sorted(files):
            full_path = os.path.join(root, file)
            file_hash = sha256_file(full_path)
            file_size = os.path.getsize(full_path)
            ext = file.split(".")[-1].lower()
            file_drive_id = drive_id_for_path_with_retry(full_path, retries=3, wait_seconds=1)

            lower_file = file.lower()
            role = "raw_extracted"

            if ext == "sav":
                role = "primary_raw_sav"
            elif ext == "pdf":
                if "diccionario" in lower_file:
                    role = "variable_dictionary_pdf"
                elif any(k in lower_file for k in ["cuestionario", "ficha", "manual", "crs04"]):
                    role = "questionnaire_pdf"
                else:
                    role = "other_pdf"

            record = {
                "file_name": file,
                "relative_path": os.path.relpath(full_path, module_folder).replace(os.sep, "/"),
                "extension": ext,
                "file_role": role,
                "drive_id": file_drive_id,
                "sha256": file_hash,
                "size_bytes": file_size,
            }
            extracted_records.append(record)

            catalog.append({
                "modulo": module_id,
                "archivo": file,
                "extension": ext,
                "file_role": role,
                "MB": file_size / (1024 * 1024),
                "drive_id": file_drive_id,
                "sha256": file_hash,
            })

    sav_files = [r for r in extracted_records if r["file_role"] == "primary_raw_sav"]
    questionnaire_pdfs = [r for r in extracted_records if r["file_role"] == "questionnaire_pdf"]
    dictionary_pdfs = [r for r in extracted_records if r["file_role"] == "variable_dictionary_pdf"]

    primary_sav = sav_files[0] if sav_files else None

    log_lines.append(f"EXTRACTED FILES COUNT: {len(extracted_records)}")
    log_lines.append(f"PRIMARY SAV: {primary_sav['file_name'] if primary_sav else 'NONE'}")
    log_lines.append(f"QUESTIONNAIRE PDF: {questionnaire_pdfs[0]['file_name'] if questionnaire_pdfs else 'NONE'}")
    log_lines.append(f"VARIABLE DICTIONARY PDF: {dictionary_pdfs[0]['file_name'] if dictionary_pdfs else 'NONE'}")
    log_lines.append(f"STATUS: SUCCESS")

    manifest.append({
        "official_microdata_portal": OFFICIAL_MICRODATA_PORTAL,
        "module_id": module_id,
        "selected_download_format": SELECTED_DOWNLOAD_FORMAT,
        "official_selected_package_type": OFFICIAL_SELECTED_PACKAGE_TYPE,
        "official_zip_file": os.path.basename(zip_path),
        "source_url": url,
        "zip_drive_id": zip_drive_id,
        "zip_sha256": zip_hash,
        "zip_size_bytes": zip_size,
        "status": "success",
        "drive_folder_id": drive_folder_id,
        "extracted_data_file": primary_sav["file_name"] if primary_sav else None,
        "extracted_data_format": "sav" if primary_sav else None,
        "questionnaire_pdf": questionnaire_pdfs[0]["file_name"] if questionnaire_pdfs else None,
        "variable_dictionary_pdf": dictionary_pdfs[0]["file_name"] if dictionary_pdfs else None,
        "extracted_file_drive_id": primary_sav["drive_id"] if primary_sav else None,
        "extracted_file_sha256": primary_sav["sha256"] if primary_sav else None,
        "extracted_file_size_bytes": primary_sav["size_bytes"] if primary_sav else None,
        "extracted_files": extracted_records,
        "alternative_formats_available": ALTERNATIVE_FORMATS_AVAILABLE,
        "alternative_formats_not_selected_reason": ALTERNATIVE_FORMATS_NOT_SELECTED_REASON,
        "timestamp": timestamp,
        "script_version": SCRIPT_VERSION,
    })

# 6. SAVE ARTIFACTS
manifest_path = os.path.join(RAW_DIR, f"ENARES_2024_STAGE1_manifest_{timestamp}.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

log_path = os.path.join(LOG_DIR, f"ENARES_2024_STAGE1_log_ingesta_{timestamp}.txt")
with open(log_path, "w") as f:
    f.write("\n".join(log_lines))

catalog_df = pd.DataFrame(catalog)
expected_catalog_columns = ["modulo", "archivo", "extension", "file_role", "MB", "drive_id", "sha256"]
catalog_df = catalog_df.reindex(columns=expected_catalog_columns)
catalog_path = os.path.join(LOG_DIR, "ENARES_2024_STAGE1_catalogo_modulos.csv")
catalog_df.to_csv(catalog_path, index=False)

print("\nIngestion loop complete. Artifacts saved.")

Starting ingestion loop...


Processing Modules: 100%|██████████| 22/22 [02:16<00:00,  6.21s/it]



Ingestion loop complete. Artifacts saved.


In [8]:
# ============================================================
# 7. FINAL VALIDATION & REPORTING (QA FOR ISSUE #4)
# ============================================================
print("\n" + "="*60)
print("STAGE 1 FINAL VALIDATION REPORT - CATALOGUE QA")
print("="*60)

# Load the catalogue back from disk to prove it was saved correctly
qa_catalog_df = pd.read_csv(catalog_path)
expected_modules_set = {f"976-Modulo{i}" for i in range(1941, 1963)}

# --- 1. Catalogue Location & Basic Stats ---
print("\n[CHECK 1] Location & Basic Stats")
print(f"Catalogue path: {catalog_path}")
print(f"Catalogue exists = {os.path.exists(catalog_path)}")
print(f"Number of rows in the catalogue: {len(qa_catalog_df)}")
unique_modules = qa_catalog_df['modulo'].nunique()
print(f"Number of unique modules represented: {unique_modules}")

# --- 2. Required Columns Check ---
print("\n[CHECK 2] Required Columns Validation")
required_cols = ["modulo", "archivo", "extension", "file_role", "MB", "drive_id", "sha256"]
missing_cols = [col for col in required_cols if col not in qa_catalog_df.columns]
if missing_cols:
    print(f"WARNING: Missing required columns: {missing_cols}")
else:
    print("All required columns are present.")
    print(f"Columns confirmed: {list(qa_catalog_df.columns)}")

# --- 3. 22 Expected Modules Representation ---
print("\n[CHECK 3] Expected Modules Representation")
represented_modules = set(qa_catalog_df['modulo'].unique())
missing_catalog_modules = expected_modules_set - represented_modules
print(f"Expected modules = 22")
print(f"Modules represented in catalogue = {len(represented_modules)}")
print(f"Missing modules = {len(missing_catalog_modules)}")
if missing_catalog_modules:
    print(f" Missing module IDs: {sorted(missing_catalog_modules)}")
else:
    print(" All 22 expected modules are catalogued.")

# --- 4. Extracted .sav Files Check ---
print("\n[CHECK 4] Primary .sav Files Verification")
sav_df = qa_catalog_df[qa_catalog_df['extension'] == 'sav']
print(f"Number of .sav files in catalogue: {len(sav_df)}")
modules_with_sav = sav_df['modulo'].nunique()
print(f"Modules with at least one .sav: {modules_with_sav}")
modules_missing_sav = expected_modules_set - set(sav_df['modulo'].unique())

if modules_missing_sav:
    print(f"Modules missing .sav: {len(modules_missing_sav)}")
    print(f"IDs of modules missing .sav: {sorted(modules_missing_sav)}")
else:
    print("All modules contain at least one .sav file.")
print("\nList of .sav files by module:")
print(sav_df[['modulo', 'archivo', 'file_role']].to_string(index=False))

# --- 5. PDF Files Classification Review ---
print("\n[CHECK 5] PDF Files & Classification Review")
pdf_df = qa_catalog_df[qa_catalog_df['extension'] == 'pdf']
if not pdf_df.empty:
    print(f"Total PDFs found: {len(pdf_df)}")
    # Print exactly the columns requested by the reviewer
    print(pdf_df[['modulo', 'archivo', 'extension', 'file_role', 'MB', 'drive_id', 'sha256']].to_string(index=False))
else:
    print(" WARNING: No PDF files found in catalogue.")

# --- 6. SHA-256 Completeness ---
print("\n[CHECK 6] SHA-256 Completeness Check")
print(f"Total catalogue rows: {len(qa_catalog_df)}")
missing_sha = qa_catalog_df[qa_catalog_df['sha256'].isna() | (qa_catalog_df['sha256'] == '')]
print(f"Rows with missing/empty SHA-256: {len(missing_sha)}")
if not missing_sha.empty:
    print("Affected files missing SHA-256:")
    print(missing_sha[['modulo', 'archivo']].to_string(index=False))
else:
    print(" 100% SHA-256 completeness achieved.")

# --- 7. Drive ID Completeness ---
print("\n[CHECK 7] Drive ID Completeness Check")
missing_drive_ids = qa_catalog_df[qa_catalog_df['drive_id'].isna() | (qa_catalog_df['drive_id'] == '')]
print(f"Rows with missing Drive ID: {len(missing_drive_ids)}")
if not missing_drive_ids.empty:
    print("FLAG: The following files are missing Drive IDs (Likely due to Drive API sync delay).")
    print("This does not mean the file is missing from Drive, only that the API took too long to index it during the run.")
    print(missing_drive_ids[['modulo', 'archivo', 'file_role']].to_string(index=False))
else:
    print(" All files successfully registered with a Drive API ID.")


STAGE 1 FINAL VALIDATION REPORT - CATALOGUE QA

[CHECK 1] Location & Basic Stats
Catalogue path: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE1_catalogo_modulos.csv
Catalogue exists = True
Number of rows in the catalogue: 66
Number of unique modules represented: 22

[CHECK 2] Required Columns Validation
All required columns are present.
Columns confirmed: ['modulo', 'archivo', 'extension', 'file_role', 'MB', 'drive_id', 'sha256']

[CHECK 3] Expected Modules Representation
Expected modules = 22
Modules represented in catalogue = 22
Missing modules = 0
 All 22 expected modules are catalogued.

[CHECK 4] Primary .sav Files Verification
Number of .sav files in catalogue: 22
Modules with at least one .sav: 22
All modules contain at least one .sav file.

List of .sav files by module:
        modulo             archivo       file_role
976-Modulo1941 01_CRS01_CAP100.sav primary_raw_sav
976-Modulo1942 02_CRS01_CAP200.sav primary_raw_sav
976-Modulo1943 03_CRS01_